# SGLang 0.5.20 wheelhouse for Nemotron 3.5 Super VL

The RTX Pro 6000 runs with **internet off**, so the Nemotron server has to arrive as wheels in a dataset. This
notebook runs **with internet on**. It downloads SGLang 0.5.20 and everything it needs, writes `requirements.lock`,
installs the lot offline into a fresh venv the same way `harness/llm/sglang.py` does on the GPU, checks it, and
leaves it in `/kaggle/working/sglang0520wheels`.

Why 0.5.20: it is the version the model card names, and the first that knows this checkpoint by its own
architecture name (`NemotronH_Omni_Reasoning_V3`). On the 0.5.19 Pennyroyal wheels we had to rename it to the Nano
Omni class. Under that name SGLang:
- did not know that `qkv_proj` packs `q_proj`, `k_proj` and `v_proj`, so the FP8 scales of the 8 attention layers
  were dropped (the `qkqkv_proj ... not found` lines);
- built a whole second model for MTP instead of the MTP layer alone (the out-of-memory error).

0.5.20 has the right class for both. torch is 2.13.0 (CUDA 13.0, from PyPI), the same as Pennyroyal.

**How to use**
1. Settings: Internet **on**. Accelerator: **none** (CPU) is enough. A GPU only adds the optional import check at
   the end; do not spend the RTX Pro 6000 on this. Then Save Version, "Save & Run All".
2. When it finishes, the last cell prints `ALL CHECKS PASSED`. Make a dataset from the output folder
   `sglang0520wheels/` (it holds `wheels/`, `requirements.lock`, `manifest.json`) named
   **`banwait13/sglang0520wheels`**: that is the name `harness/llm/nemotron.py` looks for.
3. Attach that dataset to `kaggle/experimental/sarbloh_experimentation.ipynb`.

Kaggle drops `+` from uploaded file names (`torch-2.13.0cu130-...`); `sglang.relink()` puts it back at install
time, so nothing has to be renamed by hand.

## CONFIG

In [ ]:
CONFIG = {
    "packages": [
        "sglang==0.5.20",
        # the CUDA 13.0 compiler: flashinfer and SGLang compile kernels at start, and harness/llm/sglang.py uses
        # nvidia/cu13 from these wheels as CUDA_HOME (same versions as the Pennyroyal wheelhouse)
        "nvidia-cuda-nvcc==13.0.88", "nvidia-cuda-crt==13.0.88", "nvidia-nvvm==13.0.88", "nvidia-cuda-cccl==13.0.85",
        "uv==0.12.20",   # the offline installer; harness/llm/sglang.py takes the binary out of this wheel
    ],
    # prebuilt flashinfer kernels, so fewer compiles at server start; a missing one is reported, not fatal
    "optional_packages": ["flashinfer-cubin==0.6.18", "flashinfer-jit-cache==0.6.18+cu130"],
    "extra_index_urls": ["https://flashinfer.ai/whl/", "https://flashinfer.ai/whl/cu130/"],
    # what the lock must pin (the versions SGLang 0.5.20 asks for); the build stops if any differs
    "expect": {"sglang": "0.5.20", "torch": "2.13.0", "flashinfer-python": "0.6.18", "sglang-kernel": "0.4.7",
               "transformers": "5.12.1"},
    "out_dir": "/kaggle/working/sglang0520wheels",
}

## 1. Build environment (the wheels must match the GPU image: Python 3.12, Linux x86_64)

In [ ]:
import json, os, platform, re, shutil, subprocess, sys, time, zipfile
from pathlib import Path

OUT = Path(CONFIG["out_dir"])
WHEELS = OUT / "wheels"
print("this kernel: python", sys.version.split()[0], "|", platform.platform(), "| glibc", platform.libc_ver())
assert platform.machine() == "x86_64"
print(shutil.which("nvidia-smi") and subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                     "--format=csv"], capture_output=True, text=True).stdout or "no GPU (fine for building)")

# The GPU image runs Python 3.12, and this kernel's Python may differ (the CPU image does). So the wheels are built
# from a separate Python 3.12 that uv downloads, with pip in it; nothing below uses this kernel's own Python.
BUILD = Path("/tmp/build-py312")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "uv==0.12.20"],
               check=True)
import importlib; importlib.invalidate_caches()
import uv as _uv
UV_BOOT = _uv.find_uv_bin()
shutil.rmtree(BUILD, ignore_errors=True)
subprocess.run([UV_BOOT, "venv", "--python", "3.12", "--seed", str(BUILD)], check=True)
BUILD_PY = str(BUILD / "bin" / "python")
subprocess.run([BUILD_PY, "-m", "pip", "install", "--quiet", "--upgrade", "pip"], check=True)
BUILD_VER = subprocess.run([BUILD_PY, "-c", "import sys; print(sys.version.split()[0])"],
                           capture_output=True, text=True, check=True).stdout.strip()
print("build python", BUILD_VER)
assert BUILD_VER.startswith("3.12."), "the GPU image runs Python 3.12; the build Python must match"


def pip(*args, check=True, index=True):
    cmd = [BUILD_PY, "-m", "pip", *args, "--disable-pip-version-check"]
    if index:
        for url in CONFIG["extra_index_urls"]:
            cmd += ["--extra-index-url", url]
    print("$ pip", " ".join(args)[:300], flush=True)
    return subprocess.run(cmd, check=check)

## 2. Download every wheel

`pip wheel` takes a ready wheel where one exists and builds one from source where a package only ships source,
so the GPU never has to build anything.

In [ ]:
shutil.rmtree(OUT, ignore_errors=True)
WHEELS.mkdir(parents=True)
t = time.time()
pip("wheel", "--wheel-dir", str(WHEELS), "--prefer-binary", *CONFIG["packages"])
optional_ok = []
for pkg in CONFIG["optional_packages"]:
    if pip("wheel", "--wheel-dir", str(WHEELS), "--prefer-binary", "--no-deps", pkg, check=False).returncode == 0:
        optional_ok.append(pkg)
    else:
        print(f"optional package unavailable, skipped: {pkg}")
wheels = sorted(WHEELS.glob("*.whl"))
size_gb = sum(w.stat().st_size for w in wheels) / 1e9
print(f"{len(wheels)} wheels, {size_gb:.1f} GB, {time.time() - t:.0f}s")

## 3. Write `requirements.lock`: the exact set, resolved **offline** from the wheels only

In [ ]:
report = Path("/tmp/resolve-report.json")
pip("install", "--dry-run", "--ignore-installed", "--no-index", "--find-links", str(WHEELS),
    "--report", str(report), "--quiet", *CONFIG["packages"], *optional_ok, index=False)
items = json.loads(report.read_text())["install"]
lock = sorted(f"{i['metadata']['name']}=={i['metadata']['version']}" for i in items)
(OUT / "requirements.lock").write_text("\n".join(lock) + "\n")
pins = {l.split("==")[0].lower().replace("_", "-"): l.split("==")[1] for l in lock}
print(f"requirements.lock: {len(lock)} pins")
for name, want in CONFIG["expect"].items():
    got = pins.get(name, "")
    print(f"  {name}=={got}")
    assert got.split("+")[0] == want, f"{name}: lock has {got!r}, SGLang 0.5.20 needs {want}"

## 4. Prove it: install offline exactly as `harness/llm/sglang.py` does on the GPU

The same `uv` from the wheelhouse, `--no-index`, into a fresh venv, then the checks the server start depends on.

In [ ]:
PREFIX = Path("/tmp/sglang-verify")
shutil.rmtree(PREFIX, ignore_errors=True)
(PREFIX / "bin").mkdir(parents=True)
uv_whl = sorted(WHEELS.glob("uv-*.whl"))
assert len(uv_whl) == 1, f"expected one uv wheel, found {[p.name for p in uv_whl]}"
with zipfile.ZipFile(uv_whl[0]) as z:
    uv_name = [n for n in z.namelist() if Path(n).name == "uv" and not n.endswith("/")]
    assert len(uv_name) == 1
    uv = PREFIX / "bin" / "uv"
    uv.write_bytes(z.read(uv_name[0]))
uv.chmod(0o755)
venv, py = PREFIX / "venv", PREFIX / "venv" / "bin" / "python"
env = {**{k: v for k, v in os.environ.items() if k != "PYTHONPATH"}, "UV_OFFLINE": "1", "UV_PYTHON_DOWNLOADS": "never",
       "PYTHONNOUSERSITE": "1", "UV_CACHE_DIR": str(PREFIX / "uv-cache")}
t = time.time()
subprocess.run([str(uv), "venv", "--python", BUILD_PY, str(venv)], env=env, check=True)
subprocess.run([str(uv), "pip", "install", "--python", str(py), "--no-index", "--find-links", str(WHEELS),
                "-r", str(OUT / "requirements.lock")], env=env, check=True)
print(f"offline install ok in {time.time() - t:.0f}s")


def run_py(src):
    """(exit code, last stdout line, stderr tail): the last stdout line is what the probe prints."""
    res = subprocess.run([str(py), "-c", src], env=env, capture_output=True, text=True)
    last = (res.stdout.strip().splitlines() or [""])[-1]
    return res.returncode, last, res.stderr.strip()[-2000:]


CHECKS = {}
site = next(venv.glob("lib/python*/site-packages"))
CHECKS["one sglang wheel"] = len(list(WHEELS.glob("sglang-*.whl"))) == 1
CHECKS["CUDA 13 nvcc (CUDA_HOME)"] = (site / "nvidia" / "cu13" / "bin" / "nvcc").is_file()
code_, out, err = run_py("import torch; print(torch.__version__, torch.version.cuda)")
print("torch:", out, "" if code_ == 0 else err)
CHECKS["torch 2.13.0 on CUDA 13.0"] = code_ == 0 and out.startswith("2.13.0") and out.split()[-1] == "13.0"
code_, out, err = run_py("import sglang, flashinfer, transformers; "
                         "print(sglang.__version__, flashinfer.__version__, transformers.__version__)")
print("sglang, flashinfer, transformers:", out, "" if code_ == 0 else err)
CHECKS["sglang, flashinfer, transformers import"] = code_ == 0 and out.startswith("0.5.20")
# The Nemotron pieces the server needs, read from the installed source (no GPU needed for this)
src = lambda rel: (site / "sglang" / rel).read_text(encoding="utf-8") if (site / "sglang" / rel).is_file() else ""
vl, cfg = src("srt/models/nano_nemotron_vl.py"), src("srt/configs/model_config.py")
CHECKS["native Nemotron Super VL class"] = "class NemotronH_Omni_Reasoning_V3" in vl
CHECKS["qkv packing declared (FP8 attention scales load)"] = (
    "packed_modules_mapping = NemotronHForCausalLM.packed_modules_mapping" in vl)
CHECKS["MTP draft is the MTP layer only"] = ("NemotronHForCausalLMMTP" in cfg
                                            and "class NemotronHForCausalLMMTP" in src("srt/models/nemotron_h_mtp.py"))
CHECKS["nemotron_3 reasoning parser"] = '"nemotron_3"' in src("srt/parser/reasoning_parser.py")
for name, good in CHECKS.items():
    print(("PASS " if good else "FAIL ") + name)

## 5. Optional GPU check (only when this notebook has a GPU)

Loads the SGLang kernels and the Nemotron model module on the card. On a T4 this proves the import path; the real
test is the server start on the RTX Pro 6000.

In [ ]:
has_gpu = bool(shutil.which("nvidia-smi")) and subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode == 0
if has_gpu:
    code_, out, err = run_py("import torch, sgl_kernel; "
                             "from sglang.srt.models.nano_nemotron_vl import NemotronH_Omni_Reasoning_V3; "
                             "print('gpu', torch.cuda.get_device_name(0), 'sgl_kernel ok')")
    print(out, "" if code_ == 0 else err)
    CHECKS["GPU: sgl_kernel + Nemotron module import"] = code_ == 0
else:
    print("no GPU: skipped (fine)")

## 6. Manifest, clean up, verdict

In [ ]:
manifest = {"built_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "python": BUILD_VER,
            "packages": CONFIG["packages"], "optional": optional_ok,
            "pins": {k: pins.get(k) for k in CONFIG["expect"]}, "lock_pins": len(lock),
            "wheels": len(list(WHEELS.glob("*.whl"))),
            "size_gb": round(sum(w.stat().st_size for w in WHEELS.glob("*.whl")) / 1e9, 2), "checks": CHECKS}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=1))
shutil.rmtree(PREFIX, ignore_errors=True)   # the output folder keeps the wheelhouse only
print(json.dumps(manifest, indent=1))
failed = [k for k, v in CHECKS.items() if not v]
assert not failed, f"CHECKS FAILED: {failed} - do not upload this wheelhouse"
print(f"ALL CHECKS PASSED: make a dataset from {OUT} named banwait13/sglang0520wheels")